# Explorer `client_features.parquet` et premiers modèles

<a href="https://colab.research.google.com/github/m4rc0z/finnova-hackathon-NBA-backend/blob/client-features/client_features/client_features_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Une ligne par client (8 046 clients, 83 colonnes), construite à partir des données du challenge par
`scripts/build_client_features.py`. Colonnes et pièges : `NOTICE_CLIENT_FEATURES.md` (guide) et `FEATURES.md` (référence).

**Au programme**
1. Charger le fichier (Google Drive ou upload)
2. Vue d'ensemble et qualité des données : trois pièges à neutraliser
3. Exploration : population, détention de produits, structure des dépenses, corrélations, labels
4. **Segmentation** (K-means) → personas
5. **Look-alike** : « trouve les 200 clients qui ressemblent à celui-ci »
6. **Propension** : qui pourrait ouvrir un pilier 3a ? → liste de *next best actions*
7. **Détection d'anomalies** (Isolation Forest)
8. **Prédire la sortie**, et une leçon sur la fuite de données
9. Exporter les scores en parquet

**Runtime** : un runtime CPU standard suffit, l'ensemble tourne en 1 à 2 minutes. 8 000 lignes, c'est minuscule :
le GPU ou le TPU de Colab Pro+ n'apporte rien ici. Gardez les crédits pour les modèles sur les 2,3 M d'événements
ou pour les LLM.

**Données** : le parquet est dérivé des données des organisateurs. Une fois dans votre Drive, il est hébergé chez
Google.

## 0. Charger les données

- **Option A (recommandée)** : déposez `client_features.parquet` dans Google Drive, sous `MyDrive/nba-studio/`.
  Il survit aux redémarrages du runtime, et les scores exportés à la fin atterrissent au même endroit.
- **Option B** : si le fichier n'est pas trouvé, la cellule ouvre une boîte d'upload. Le fichier est alors perdu au
  prochain redémarrage.

Pour changer le chemin, modifiez `PARQUET_PATH` ci-dessous.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

IN_COLAB = "google.colab" in sys.modules
PARQUET_PATH = os.environ.get("NBA_PARQUET", "/content/drive/MyDrive/nba-studio/client_features.parquet")
RANDOM_STATE = 42

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path(PARQUET_PATH).exists():
        from google.colab import files
        print(f"{PARQUET_PATH} not found: pick client_features.parquet on your computer.")
        uploaded = files.upload()
        PARQUET_PATH = str(Path.cwd() / next(iter(uploaded)))

df = pd.read_parquet(PARQUET_PATH)
assert df["individual_id"].is_unique
print(f"{len(df):,} clients x {df.shape[1]} columns")
df.head(3).T

## 1. Vue d'ensemble

Les colonnes sont rangées par familles. Les listes ci-dessous servent dans tout le notebook.
Tout ce qui commence par `label_` est une **cible** et ne doit jamais servir de variable explicative.

In [ ]:
LABELS = [c for c in df.columns if c.startswith("label_")]
CATEGORICAL = ["sex", "canton", "nationality", "education_level", "occupation", "sector",
               "marital_status", "employment_type", "employer_sector", "employer_size_band"]
HOLDINGS = [c for c in df.columns if c.startswith("has_") and c not in ("has_13th_salary", "has_annual_bonus")]
SPEND = [c for c in df.columns if c.startswith("spend_") and c != "spend_total_monthly_chf"]
BIG5 = [c for c in df.columns if c.startswith("big5_")]

overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null_%": (df.notna().mean() * 100).round(1),
    "n_unique": df.nunique(),
})
print("Columns with missing values (the full table is in `overview`):")
overview[overview["non_null_%"] < 100].sort_values("non_null_%")

## 2. Qualité des données : trois pièges à neutraliser avant tout modèle

1. **Anomalie hypothécaire** : pour 12 propriétaires, la mensualité vaut toute la dette restante. C'est un défaut de
   la simulation, et il produit des découverts de plusieurs millions de CHF.
2. **Clients sans transaction** sur la fenêtre (surtout des mineurs) : leurs flux mensuels sont `NaN`.
3. **Clients sortis** (`label_exited`) : leur ligne montre l'état *après* la sortie. Ils n'ont plus aucun compte
   ouvert, donc tous leurs `has_*` sont faux.

La population de travail `active` exclut ces trois cas. Sauf mention contraire, les sections suivantes portent
sur elle.

In [ ]:
df["is_mortgage_anomaly"] = (df["mortgage_monthly_chf"] / df["mortgage_outstanding_chf"]) > 0.5
print("Mortgage anomalies (monthly payment ~ whole outstanding):", int(df["is_mortgage_anomaly"].sum()))
print("Clients without any transaction in the window:", int((df["active_months"] == 0).sum()))
print("Exited clients:", int(df["label_exited"].sum()),
      "| of which with an open account:", int((df["label_exited"] & (df["n_accounts_open"] > 0)).sum()))

active = df[~df["label_exited"] & (df["active_months"] > 0) & ~df["is_mortgage_anomaly"]].copy()
active["age_band"] = pd.cut(active["age_years"], [0, 18, 26, 36, 51, 66, 120], right=False,
                            labels=["<18", "18-25", "26-35", "36-50", "51-65", "65+"])
print(f"\nWorking population 'active': {len(active):,} clients")

## 3. Exploration

### 3.1 Qui sont les clients ?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.histplot(active["age_years"], bins=40, ax=axes[0, 0]).set(title="Age", xlabel="years")
active["employment_type"].fillna("unknown").value_counts().plot.barh(ax=axes[0, 1], title="Employment type")
sns.histplot(active.loc[active["income_monthly_chf"] > 0, "income_monthly_chf"], bins=60, log_scale=True,
             ax=axes[1, 0]).set(title="Monthly income, CHF (> 0, log scale)")
active["canton"].value_counts().head(12).plot.bar(ax=axes[1, 1], title="Top 12 cantons")
plt.tight_layout()
plt.show()

### 3.2 Détention de produits par âge

Le point de départ de toute *next best action* : qui a quoi.

In [ ]:
penetration = active.groupby("age_band", observed=True)[HOLDINGS].mean().mul(100)
plt.figure(figsize=(12, 4))
sns.heatmap(penetration, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "% of clients"})
plt.title("Product penetration by age band (% of active clients)")
plt.show()

### 3.3 Structure des dépenses

La part de chaque catégorie dans la dépense mensuelle révèle le mode de vie mieux que les montants bruts.

In [ ]:
shares = (active[SPEND].div(active["spend_total_monthly_chf"], axis=0)
          .replace([np.inf, -np.inf], np.nan).fillna(0))
shares.columns = [c.removeprefix("spend_").removesuffix("_monthly_chf") for c in SPEND]

mix = shares.groupby(active["age_band"], observed=True).mean()
mix.plot.barh(stacked=True, figsize=(12, 4), colormap="tab10", title="Spending mix by age band")
plt.legend(bbox_to_anchor=(1, 1))
plt.xlabel("share of monthly spend")
plt.show()

plt.figure(figsize=(8, 6))
sns.scatterplot(data=active[active["income_monthly_chf"] > 0], x="income_monthly_chf", y="spend_total_monthly_chf",
                hue="employment_type", alpha=0.4, s=12)
plt.xscale("log")
plt.yscale("log")
plt.title("Income vs spend (CHF/month)")
plt.show()

### 3.4 Corrélations

Corrélation de Spearman (sur les rangs), robuste aux queues très longues des montants en CHF.

In [ ]:
num_cols = ["age_years", "income_monthly_chf", "salary_monthly_chf", "balance_total_chf", "balance_savings_chf",
            "min_monthly_total_balance_chf", "months_in_overdraft", "rent_monthly_chf",
            "savings_transfer_monthly_chf", "pillar3a_contribution_monthly_chf", "spend_total_monthly_chf",
            "savings_rate", "risk_appetite", "wallet_share", "health_score", "n_accounts_open"] + BIG5
plt.figure(figsize=(12, 10))
sns.heatmap(active[num_cols].corr(method="spearman"), cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Spearman correlations (active clients)")
plt.show()

### 3.5 Les labels

Sur l'ensemble des clients : taux de sortie, et répartition des sorties par période.

In [ ]:
print(df[LABELS].drop(columns="label_exit_period").mean().mul(100).round(2).rename("% of all clients"))
(df[df["label_exited"]]
   .assign(other=lambda d: ~d["label_churn_with_reason"] & ~d["label_is_death"])
   .groupby("label_exit_period")[["label_churn_with_reason", "label_is_death", "other"]].sum()
   .plot.bar(stacked=True, figsize=(10, 3), title="Exits per period"))
plt.show()

## 4. Segmentation : des personas à partir des comportements

On décrit chaque client par ses **parts de dépense**, ses montants (revenu, soldes, dépense, loyer) passés en
log signé, son âge, son taux d'épargne, ses mois de découvert et son nombre de comptes. On standardise, puis on
cherche le nombre de groupes *k* qui maximise le score de silhouette.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


def slog(x):
    """Signed log: keeps the sign of negative balances and compresses the heavy CHF tails."""
    return np.sign(x) * np.log1p(np.abs(x))


X_seg = pd.concat([
    shares.add_prefix("share_"),
    slog(active[["income_monthly_chf", "balance_total_chf", "balance_savings_chf",
                 "spend_total_monthly_chf", "rent_monthly_chf"]]).add_prefix("slog_"),
    active[["age_years", "months_in_overdraft", "n_accounts_open"]],
    active[["savings_rate"]].fillna(0),
], axis=1)
Z = StandardScaler().fit_transform(X_seg)

silhouette = {}
for k in range(3, 9):
    labels_k = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(Z)
    silhouette[k] = silhouette_score(Z, labels_k, sample_size=3000, random_state=RANDOM_STATE)
K = max(silhouette, key=silhouette.get)   # override by hand if you want more or fewer personas
print({k: round(v, 3) for k, v in silhouette.items()}, "-> K =", K)

In [ ]:
kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(Z)
active["segment"] = kmeans.labels_
pca = PCA(n_components=2, random_state=RANDOM_STATE).fit(Z)
P = pca.transform(Z)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=P[:, 0], y=P[:, 1], hue=active["segment"], palette="tab10", s=10, alpha=0.6)
plt.title(f"Segments in the first two principal components ({pca.explained_variance_ratio_.sum():.0%} of variance)")
plt.show()

profile = active.groupby("segment").agg(
    clients=("individual_id", "size"),
    age=("age_years", "median"),
    income=("income_monthly_chf", "median"),
    balance=("balance_total_chf", "median"),
    spend=("spend_total_monthly_chf", "median"),
    savings_rate=("savings_rate", "median"),
    overdraft_months=("months_in_overdraft", "mean"),
    retired=("employment_type", lambda s: (s == "retired").mean()),
    self_employed=("is_self_employed_with_business_account", "mean"),
    has_pillar3a=("has_pillar3a", "mean"),
    has_investment=("has_investment", "mean"),
    owns_property=("owns_property", "mean"),
)
profile

Ce qui distingue chaque segment : la part de dépense du segment divisée par la part moyenne. Au-dessus de 1, la
catégorie est sur-représentée. Avec le tableau ci-dessus, cela suffit pour nommer les personas, par exemple
« retraités à forte épargne » ou « jeunes actifs, sorties et transport ».

In [ ]:
over = shares.groupby(active["segment"]).mean() / shares.mean()
plt.figure(figsize=(11, 0.6 * K + 1.5))
sns.heatmap(over, annot=True, fmt=".2f", cmap="RdBu_r", center=1)
plt.title("Spend share relative to the average client (> 1 = over-represented)")
plt.show()

## 5. Look-alike : « ce client est important, trouve les 200 qui lui ressemblent »

Plus proches voisins dans le même espace standardisé que la segmentation. Le client ancre est, par défaut, celui qui
a le plus gros portefeuille d'investissement. Mettez un `individual_id` dans `ANCHOR_ID` pour en choisir un autre.
Les sosies qui n'ont **pas encore** de compte d'investissement sont des candidats naturels pour une offre de conseil.

In [ ]:
from sklearn.neighbors import NearestNeighbors

ANCHOR_ID = None
anchor_pos = (int(np.argmax(active["balance_investment_chf"].to_numpy())) if ANCHOR_ID is None
              else int(np.flatnonzero(active["individual_id"].to_numpy() == ANCHOR_ID)[0]))
nn = NearestNeighbors(n_neighbors=201).fit(Z)
_, idx = nn.kneighbors(Z[[anchor_pos]])
anchor = active.iloc[anchor_pos]
lookalikes = active.iloc[idx[0][1:]]

cols = ["age_years", "income_monthly_chf", "balance_total_chf", "spend_total_monthly_chf", "savings_rate",
        "has_pillar3a", "has_investment", "owns_property"]
compare = pd.DataFrame({
    "anchor": anchor[cols].astype(float),
    "200 look-alikes": lookalikes[cols].astype(float).median().where(lookalikes[cols].dtypes != bool,
                                                                     lookalikes[cols].mean()),
    "all active": active[cols].astype(float).median().where(active[cols].dtypes != bool, active[cols].mean()),
})
print(f"Anchor {anchor['individual_id']} (segment {anchor['segment']}); booleans shown as rates")
display(compare)

targets = lookalikes[~lookalikes["has_investment"]]
print(f"{len(targets)} of the 200 look-alikes have no investment account yet -> advisory-mandate candidates")
targets[["individual_id", "age_years", "income_monthly_chf", "balance_total_chf", "balance_savings_chf"]].head(10)

## 6. Propension : qui pourrait ouvrir un pilier 3a ?

On apprend à distinguer les adultes qui ont un pilier 3a de ceux qui n'en ont pas. Les clients **sans** 3a mais avec
un **score élevé** ressemblent à ceux qui en ont un : ce sont les candidats d'une *next best action*.

- **Modèle** : `HistGradientBoostingClassifier` de scikit-learn. Il est rapide, gère nativement les valeurs
  manquantes et les catégories, et marche partout sans installation.
- **Évaluation** : validation croisée à 5 plis. Chaque client est noté par un modèle qui ne l'a pas vu
  à l'entraînement (prédictions *out-of-fold*).
- **Fuite à retirer** : les colonnes qui trahissent la cible. Ce sont la contribution et le solde 3a, mais aussi les
  soldes *totaux*, qui incluent le solde 3a, et le nombre de comptes, qui compte le compte 3a.
- **Population éligible** : le 3a suppose un revenu professionnel. On se limite donc aux salariés et indépendants
  de 18 à 64 ans. Sans ce filtre, le modèle recommande des retraités, qui ne peuvent pas cotiser. C'est
  exactement le rôle d'un garde-fou dans une NBA.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split

ALWAYS_DROP = ["individual_id", "city", "first_account_period", "onboarded_period",
               "is_mortgage_anomaly", "segment", "age_band"] + LABELS


def model_matrix(frame, drop=(), keep=None):
    """Feature matrix: labels and ids removed, categoricals as pandas 'category', booleans as 0/1."""
    X = frame[keep].copy() if keep else frame.drop(columns=[c for c in ALWAYS_DROP + list(drop) if c in frame])
    for c in X.columns:
        if c in CATEGORICAL:
            X[c] = X[c].astype("category")
        elif X[c].dtype == bool:
            X[c] = X[c].astype(int)
    return X


def make_model():
    return HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, categorical_features="from_dtype",
                                          random_state=RANDOM_STATE)


def oof_scores(X, y, name):
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    oof = cross_val_predict(make_model(), X, y, cv=cv, method="predict_proba")[:, 1]
    print(f"{name}: n={len(y):,} | base rate {y.mean():.1%} | "
          f"ROC-AUC {roc_auc_score(y, oof):.3f} | PR-AUC {average_precision_score(y, oof):.3f}")
    return oof


adults = active[active["age_years"] >= 18].copy()
eligible = adults[adults["employment_type"].isin(["employed", "self_employed"]) & (adults["age_years"] < 65)].copy()
# total balances include the 3a balance itself, account counts include the 3a account
PILLAR3A_LEAKS = ["pillar3a_contribution_monthly_chf", "balance_pillar3a_chf", "balance_total_chf",
                  "min_monthly_total_balance_chf", "n_accounts_open", "n_accounts_closed"]
X = model_matrix(eligible, ["has_pillar3a"] + PILLAR3A_LEAKS)
y = eligible["has_pillar3a"].astype(int).to_numpy()
eligible["p_pillar3a"] = oof_scores(X, y, "has_pillar3a (employed / self-employed, 18-64)")

**Lecture business : le lift par décile.** On trie les clients par score et on les coupe en 10 groupes. Si le modèle
est utile, le taux de détention grimpe fortement vers les déciles hauts. Les variables les plus importantes
(importance par permutation, mesurée sur un jeu de test) disent *pourquoi*.

In [ ]:
deciles = pd.qcut(eligible["p_pillar3a"].rank(method="first"), 10, labels=range(1, 11))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
eligible.groupby(deciles, observed=True)["has_pillar3a"].mean().mul(100).plot.bar(
    ax=axes[0], title="Pillar 3a holding rate by score decile (10 = highest)", ylabel="%")
axes[0].axhline(eligible["has_pillar3a"].mean() * 100, ls="--", c="grey")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
model = make_model().fit(X_tr, y_tr)
imp = permutation_importance(model, X_te, y_te, scoring="roc_auc", n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
pd.Series(imp.importances_mean, index=X.columns).sort_values().tail(15).plot.barh(
    ax=axes[1], title="Top 15 features (permutation importance, drop in ROC-AUC)")
plt.tight_layout()
plt.show()

print(pd.crosstab(pd.cut(eligible["age_years"], [18, 25, 26, 30, 65], right=False), eligible["has_pillar3a"]))
print()
print(pd.crosstab(eligible["is_onboarded_during_run"], eligible["has_pillar3a"]))

**Un score trop beau doit toujours faire chercher une fuite.** Une première version de ce modèle laissait les soldes
totaux parmi les variables, alors qu'ils contiennent le solde 3a : c'était une fuite, retirée ci-dessus.

Le ROC-AUC reste pourtant proche de 0,99, parce que la simulation attribue le 3a presque par règle (tableaux
ci-dessus) : personne n'en a avant 25 ans, presque tous les actifs présents depuis le début en ont un, et les
clients arrivés en cours de route n'en ont presque jamais. Le modèle retrouve cette règle par l'âge et le solde
d'épargne. Sur des données réelles, un score pareil devrait faire chercher une autre fuite.

Les candidats intéressants sont donc les clients qui **n'ont pas** de 3a alors que tous leurs pairs en ont un.
C'est la matière d'une campagne ou d'un briefing conseiller.

Plus bas, la même mécanique sur une cible plus rare et moins déterministe : le compte d'investissement.

In [ ]:
candidates = eligible[~eligible["has_pillar3a"]].nlargest(20, "p_pillar3a")
display(candidates[["individual_id", "p_pillar3a", "age_years", "employment_type", "is_onboarded_during_run",
                    "income_monthly_chf", "balance_savings_chf", "savings_rate", "segment"]])

X_inv = model_matrix(adults, ["has_investment", "balance_investment_chf", "n_accounts_open", "n_accounts_closed"])
adults["p_investment"] = oof_scores(X_inv, adults["has_investment"].astype(int).to_numpy(), "has_investment")

## 7. Détection d'anomalies (Isolation Forest)

On cherche les clients dont les montants sortent de l'ordinaire, sans rien dire au modèle. Cette fois, on **garde**
les anomalies hypothécaires. Si l'algorithme les remonte tout seul, c'est un bon contrôle de bon sens, et une
démonstration utile : un outil de ce type repère aussi les défauts d'un jeu de données.

In [ ]:
from sklearn.ensemble import IsolationForest

pool = df[~df["label_exited"] & (df["active_months"] > 0)].copy()
money = ["income_monthly_chf", "balance_total_chf", "balance_checking_chf", "min_monthly_total_balance_chf",
         "rent_monthly_chf", "mortgage_payment_monthly_chf", "spend_total_monthly_chf",
         "savings_transfer_monthly_chf", "months_in_overdraft"]
Xa = StandardScaler().fit_transform(slog(pool[money].fillna(0)))
iso = IsolationForest(n_estimators=300, random_state=RANDOM_STATE).fit(Xa)
pool["anomaly_score"] = -iso.score_samples(Xa)

top = pool.nlargest(50, "anomaly_score")
print(f"Known mortgage anomalies among the 50 most anomalous clients: "
      f"{int(top['is_mortgage_anomaly'].sum())} / {int(pool['is_mortgage_anomaly'].sum())}")
top[["individual_id", "anomaly_score", "is_mortgage_anomaly", "balance_checking_chf",
     "mortgage_payment_monthly_chf", "income_monthly_chf", "months_in_overdraft"]].head(15)

## 8. Prédire la sortie, et la leçon de la fuite de données

Cible : `label_churn_with_reason`, les 278 clients partis avec un motif. Les décès sont exclus et on se limite aux
adultes.

**Modèle naïf** : toutes les colonnes. Le score sera quasi parfait, mais c'est un leurre. La ligne d'un client sorti
décrit son état *après* la sortie : plus de compte ouvert, des `has_*` à faux, moins de mois actifs. Le modèle lit la
réponse au lieu de la prédire.

In [ ]:
pop = df[(df["age_years"] >= 18) & ~df["label_is_death"]].copy()
y_churn = pop["label_churn_with_reason"].astype(int).to_numpy()
_ = oof_scores(model_matrix(pop), y_churn, "churn, naive (all columns)")

**Modèle honnête** : seulement des attributs qui ne changent pas au moment de la sortie (profil, personnalité,
santé). Le score est plus bas, mais c'est le vrai signal disponible dans ce fichier.

La bonne méthode consiste à reconstruire les variables à une date *antérieure* à la sortie (par exemple 3 mois
avant) à partir des événements bruts, puis à prédire la sortie dans les mois suivants. Le script de construction
fixe aujourd'hui la période de référence à 24301 : il faudrait la rendre paramétrable.

In [ ]:
STATIC = ["age_years", "sex", "canton", "nationality", "education_level", "occupation", "marital_status",
          "employment_type", "risk_appetite", "wallet_share", "health_score", "n_health_conditions",
          "is_onboarded_during_run"] + BIG5
pop["p_churn_static"] = oof_scores(model_matrix(pop, keep=STATIC), y_churn, "churn, static profile only")
_ = oof_scores(model_matrix(pop, keep=[c for c in STATIC if c != "risk_appetite"]), y_churn,
               "churn, static without risk_appetite")
print("\nMedian risk_appetite by churn label:", pop.groupby("label_churn_with_reason")["risk_appetite"].median().to_dict())

model_s = make_model().fit(model_matrix(pop, keep=STATIC), y_churn)
imp_s = permutation_importance(model_s, model_matrix(pop, keep=STATIC), y_churn, scoring="roc_auc",
                               n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
pd.Series(imp_s.importances_mean, index=STATIC).sort_values().plot.barh(
    figsize=(8, 5), title="Churn, static model: permutation importance (in-sample, indicative)")
plt.show()

**À retenir** : `risk_appetite` porte l'essentiel du signal (voir les médianes et le score sans cette variable).
Avec un instantané, impossible de savoir si c'est une **cause** (les profils à risque partent) ou une **conséquence**
(l'attribut a été mis à jour au moment du départ, donc une fuite). C'est une question à poser aux organisateurs
avant d'utiliser ce modèle.

## 9. Exporter les scores

Un parquet `nba_scores.parquet` à côté du fichier d'entrée (donc dans votre Drive avec l'option A), une ligne par
client : segment, propensions, score d'anomalie, score de sortie. Les valeurs sont vides pour les clients hors de la
population de chaque modèle. Ce sont exactement des colonnes qu'un studio NBA peut consommer.

In [ ]:
scores = (df[["individual_id"]]
          .merge(active[["individual_id", "segment"]], how="left")
          .merge(eligible[["individual_id", "p_pillar3a"]], how="left")
          .merge(adults[["individual_id", "p_investment"]], how="left")
          .merge(pool[["individual_id", "anomaly_score"]], how="left")
          .merge(pop[["individual_id", "p_churn_static"]], how="left"))
OUT_PATH = Path(os.environ.get("NBA_OUT_DIR", Path(PARQUET_PATH).parent)) / "nba_scores.parquet"
scores.to_parquet(OUT_PATH, index=False)
print(f"{OUT_PATH}: {scores.shape[0]:,} rows x {scores.shape[1]} columns")
scores.describe()

## 10. Pistes pour la suite

- **Personas** : nommer les segments et les relier aux produits du catalogue (quel segment sous-détient quoi).
- **Sortie** : paramétrer la période de référence du script de construction, pour apprendre sur des variables
  observées *avant* la sortie.
- **Séquences** : descendre au niveau des 2,3 M d'événements (commerçants, codes MCC, régularité des flux). C'est
  là que la puissance de Colab Pro+ devient utile.
- **Texte** : les `churn_reason` (allemand, 278 textes) ne sont pas dans ce parquet. Un LLM pourrait les classer
  en motifs (frais, service, néobanque…) pour expliquer les sorties.
- **Explicabilité** : `pip install shap` pour des explications client par client (pourquoi ce score ?), utiles
  dans un briefing conseiller.